# In vivo gonad reference, 6 to 9 weeks

Data: https://cellxgene.cziscience.com/collections/661a402a-2a5a-4c71-9b05-b346c57bc451

Creates a female and a male object for a LIANA analysis with PGCs as the target
cell type.

Three files are needed. `human_female` and `human_male` hold every cell type,
which is what a cell communication analysis needs. `human_germ` holds the
germline labels, which the whole tissue files do not have.

In [ ]:
import os, re
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

DATA_DIR = "data/garcia-alonso/"
OUTDIR = "results_gaal_invivo"
os.makedirs(OUTDIR, exist_ok=True)

# window in DAYS post fertilisation, not weeks
DAY_MIN, DAY_MAX = 40.0, 63.0

PGC_LABEL = "primordial germ cell"
N_HVG, N_PCS, RES = 2000, 30, 0.8

USE_SCVI = None   # set in the representation cell below, after the check

sc.settings.verbosity = 1

## 1. Load and inspect

Prints what is in each file: the gene names, where the counts are, and every
obs column with fewer than 60 levels.
We need `donor_id`, `sample`, `development_stage`, `cell_type` and `lineageSomatic`.

In [ ]:
files = {"female": "human_female.h5ad",
         "male": "human_male.h5ad",
         "germ": "human_germ.h5ad"}
objs = {k: sc.read_h5ad(os.path.join(DATA_DIR, v)) for k, v in files.items()}

def matrix_summary(M):
    vals = M.data if sp.issparse(M) else np.asarray(M).ravel()
    if vals.size == 0:
        return 0.0, True
    return float(vals.max()), bool(np.allclose(vals, np.round(vals)))


def describe(name, a):
    print(f"=== {name}: {a.n_obs:,} cells x {a.n_vars:,} genes ===")
    print("  var_names:", list(a.var_names[:2]),
          "| var cols:", list(a.var.columns)[:6])

    mx, ints = matrix_summary(a.X)
    print(f"  .X max {mx:.2f}, integers: {ints}")

    if a.raw is not None:
        mx, ints = matrix_summary(a.raw.X)
        print(f"  .raw max {mx:.2f}, integers: {ints}")
    else:
        print("  .raw: absent")

    print(f"  layers {list(a.layers)} | obsm {list(a.obsm)}")

    for c in a.obs.columns:
        v = a.obs[c]
        if v.dtype.name in ("category", "object") and v.nunique() < 60:
            print(f"    {c:32s} {v.nunique():3d}  {list(v.astype(str).unique())[:6]}")
    wide = [c for c in a.obs.columns
            if a.obs[c].dtype.name in ("category", "object")
            and a.obs[c].nunique() >= 60]
    if wide:
        print(f"    (not shown, 60+ levels: {wide})")
    print()


for k, a in objs.items():
    describe(k, a)

## 2. Stage

The files use two different stage labels and they are not on the same scale.
Carnegie stages convert to elapsed days.

In [ ]:
CARNEGIE_DAYS = {13: 30, 14: 33, 15: 36.5, 16: 39.5, 17: 43, 18: 46,
                 19: 49.5, 20: 52, 21: 53.5, 22: 55, 23: 58}

def stage_to_days(s):
    s = str(s)
    m = re.search(r"carnegie\s*stage\s*(\d+)", s, flags=re.I)
    if m:
        return CARNEGIE_DAYS.get(int(m.group(1)), np.nan)
    m = re.search(r"(\d+)\s*\w*\s*week", s, flags=re.I)
    if m:
        return 7.0 * int(m.group(1)) - 3.5   # midpoint of that week
    return np.nan

for a in objs.values():
    d = pd.to_numeric(a.obs["development_stage"].astype(str).map(stage_to_days),
                      errors="coerce")
    a.obs["day_pf"] = d
    a.obs["week_pf"] = d / 7.0

stages = set()
for a in objs.values():
    stages |= set(a.obs["development_stage"].astype(str).unique())
tab = pd.DataFrame({"stage": sorted(stages)})
tab["day_pf"] = tab["stage"].map(stage_to_days)
tab["week_pf"] = (tab["day_pf"] / 7).round(2)
tab["in_window"] = tab["day_pf"].between(DAY_MIN, DAY_MAX)
print(tab.sort_values("day_pf").to_string(index=False))

## 3. How many PGCs are there

In [ ]:
germ = objs["germ"]
pgc = germ[germ.obs["cell_type"].astype(str) == PGC_LABEL]
inw = pgc.obs["day_pf"].between(DAY_MIN, DAY_MAX)

print(f"PGCs in human_germ: {pgc.n_obs:,}, in window: {int(inw.sum()):,}\n")
print(pd.crosstab(pgc.obs["week_pf"].round(2), pgc.obs["sex"], margins=True))
print("\nin-window PGCs by donor:")
print(pgc.obs.loc[inw.values, "donor_id"].value_counts().to_string())

The male count is very low. Check it above before planning the male analysis.
A ligand receptor result needs somewhere around 50 target cells to mean
anything, and weeks 6 to 7 hold only a handful across both sexes.

## 4. Cell labels

The whole tissue files have `lineageSomatic`, which is a somatic annotation
with about 20 levels. It has no fine germline states, so PGC labels have to
come from `human_germ` by matching barcodes.

So the label is built from three sources in order: germ labels where the
barcode matches, `lineageSomatic` for somatic cells, and `cell_type` for
anything left over.

In [ ]:
germ_lab = pd.Series(germ.obs["cell_type"].astype(str).values,
                     index=germ.obs_names)
germ_lab = germ_lab[~germ_lab.index.duplicated()]

for sex in ["female", "male"]:
    a = objs[sex]
    joined = germ_lab.reindex(a.obs_names)
    som = a.obs["lineageSomatic"].astype(str)
    som_ok = ~som.isin(["nan", "", "None", "NA", "<NA>"])

    label = a.obs["cell_type"].astype(str).copy()        # fallback
    label[som_ok.values] = som[som_ok.values]            # somatic detail
    label[joined.notna().values] = joined[joined.notna()].values   # germ wins

    a.obs["fine_label"] = label.values
    n_germ = int(joined.notna().sum())
    n_som = int((som_ok.values & ~joined.notna().values).sum())
    n_ct = a.n_obs - n_germ - n_som
    print(f"{sex}: {n_germ:,} cells labelled from human_germ, "
          f"{n_som:,} from lineageSomatic, "
          f"{n_ct:,} from cell_type")
    print(f"   PGCs: {int((a.obs['fine_label'] == PGC_LABEL).sum()):,}")

print("\nlabels in the female object:")
print(objs["female"].obs["fine_label"].value_counts().to_string())

Look down that list for the same cell type appearing twice under different
names, for example `Endothelial` from `lineageSomatic` next to
`endothelial cell` from `cell_type`. LIANA would score those as two separate
cell types. If you see something that should be a pair, rename one to match the other.

## 5. Genes and counts

Gene names are Ensembl IDs, and LIANA needs symbols. Counts are in `.raw` and
get moved into a layer, because renaming the genes does not rename `.raw`.

Note which is which: `.X` holds log normalised values already, `.raw` holds
raw counts. That matters in the next section, where normalising `.X` a second
time would be wrong.

In [ ]:
for sex in ["female", "male"]:
    a = objs[sex]

    if a.raw is not None:
        a.layers["counts"] = a.raw[:, a.var_names].X.copy()
        a.raw = None

    syms = a.var["feature_name"].astype(str)
    a.var["ensembl_id"] = a.var_names
    a.var_names = pd.Index(syms.values)
    a.var_names_make_unique()

    a.obs["library"] = a.obs["sample"].astype(str)   # keep, do not overwrite
    a.obs["donor_sex"] = a.obs["donor_id"].astype(str) + "_" + sex

    n0 = a.n_obs
    objs[sex] = a[a.obs["day_pf"].between(DAY_MIN, DAY_MAX)].copy()
    print(f"{sex}: {n0:,} -> {objs[sex].n_obs:,} cells in window, "
          f"{objs[sex].obs['donor_id'].nunique()} donors, "
          f"{objs[sex].obs['library'].nunique()} libraries")

## 6. Combine and cluster

Sex is not corrected for. From about 7 weeks the supporting cells really are
different tissue in an ovary and a testis, so that difference is real.

Donor, stage and chemistry cannot be told apart in this dataset, because each
donor was collected at one stage and run on one chemistry. So there is no batch
variable here that could be corrected without also removing developmental time.

In [ ]:
both = ad.concat([objs["female"], objs["male"]], join="inner",
                 index_unique=None, merge="first")
for c in ["sex", "donor_sex", "library", "donor_id", "assay", "fine_label"]:
    both.obs[c] = both.obs[c].astype(str).astype("category")

print(f"{both.n_obs:,} cells x {both.n_vars:,} shared genes")
print(pd.crosstab(both.obs["donor_sex"], both.obs["week_pf"].round(2)))

# .X in the source files is ALREADY log normalised.
# Start from the counts layer instead, which is the only safe place to normalise from.
both.X = both.layers["counts"].copy()
sc.pp.normalize_total(both, target_sum=1e4)
sc.pp.log1p(both)
both.uns["log1p"] = {"base": None}
sc.pp.highly_variable_genes(both, n_top_genes=N_HVG, batch_key="sex")
sc.pp.pca(both, n_comps=N_PCS, mask_var="highly_variable")
print(f"PCA {both.obsm['X_pca'].shape}")

In [ ]:
# Further batch checks
print(pd.crosstab(both.obs["donor_id"], both.obs["assay"]))
print(pd.crosstab(both.obs["assay"], both.obs["day_pf"]))
print(pd.crosstab(both.obs["donor_id"], both.obs["day_pf"]))

### Which representation to cluster on

The files contains an `X_scVI` embedding, but each file was analysed separately, so
the two are probably from two different models. If so they are in unrelated
coordinate systems and a distance between a female and a male cell means
nothing.

The first thing to look at is the width. If the two embeddings have a different
number of dimensions then they cannot be from one model.
Only if the widths match is the distance check below worth reading, and
even then it is a hint rather than proof.

Set `USE_SCVI` in the next cell from what this prints.

If it comes out `False`, clustering goes back to ordinary PCA.
Every donor was collected at one stage and run on one chemistry, so there
is no batch variable in this dataset that could be corrected without also
removing developmental time.

In [ ]:
if not all("X_scVI" in objs[s].obsm for s in ["female", "male"]):
    print("no X_scVI in the files. Set USE_SCVI = False.")
else:
    A = np.asarray(objs["female"].obsm["X_scVI"])
    B = np.asarray(objs["male"].obsm["X_scVI"])
    print(f"widths: female {A.shape[1]}, male {B.shape[1]}")

    if A.shape[1] != B.shape[1]:
        # This is enough on its own. One model cannot give one file x latent
        # dimensions and the other y, so these are two separate models, in
        # unrelated coordinate systems. Comparing them any further would mean
        # comparing axes that have nothing to do with each other.
        print("\nDifferent widths, so these are two separately trained models.")
        print("A distance between a female cell and a male cell in this space")
        print("means nothing. Set USE_SCVI = False.")
    else:
        cent = np.linalg.norm(A.mean(0) - B.mean(0))
        spread = 0.5 * (A.std(0).mean() + B.std(0).mean())
        print(f"centroid distance {cent:.2f}, spread {spread:.2f}, "
              f"ratio {cent / spread:.2f}")
        print("\nSame width, so the two could be one model. The ratio is a")
        print("weak hint, not proof: well under 1 means the two clouds sit on")
        print("top of each other, which is what two separate models look like.")
        print("Around 1 or more is consistent with one shared model.")

    print(f"\nX_scVI in the combined object: {'X_scVI' in both.obsm}")
    if "X_scVI" not in both.obsm:
        print("It was dropped when the two files were joined, which is what")
        print("happens when the widths do not match. USE_SCVI = False is the")
        print("only option here.")

In [ ]:
# X_scVI did survive the concat, so both.obsm holds a mixture of a 60
# dimensional and a 30 dimensional model. That is not a usable embedding.
# Remove it.
if "X_scVI" in both.obsm:
    print("X_scVI in the combined object:", both.obsm["X_scVI"].shape)
    del both.obsm["X_scVI"]
    print("deleted: it mixes two separately trained models")
else:
    print("no X_scVI in the combined object, nothing to remove")

In [ ]:
USE_SCVI = False   # <-- ALREADY CHECKED - LEAVE IT FALSE

if USE_SCVI and "X_scVI" in both.obsm:
    rep = "X_scVI"
elif USE_SCVI:
    print("X_scVI did not survive the concat, falling back to PCA")
    rep = "X_pca"
else:
    rep = "X_pca"

print(f"clustering on {rep}")
sc.pp.neighbors(both, use_rep=rep)
sc.tl.leiden(both, resolution=RES, flavor="igraph", n_iterations=2,
             directed=False, key_added="leiden")
sc.tl.umap(both)
print(f"{both.obs['leiden'].nunique()} clusters")

sc.pl.umap(both, color=["leiden", "sex", "week_pf", "fine_label"],
           ncols=2, wspace=0.35, show=False)
plt.savefig(os.path.join(OUTDIR, "umap.png"), dpi=150, bbox_inches="tight")
plt.show()

Every donor comes from one stage, so a cluster made mostly of one donor is not
necessarily a batch problem, it may just be a cell type that only exists at
that stage. The table below is for looking at, not for filtering on.

In [ ]:
counts = pd.crosstab(both.obs["leiden"], both.obs["donor_sex"])
counts.to_csv(os.path.join(OUTDIR, "cells_per_cluster_per_donor.csv"))
display(counts)

In [ ]:
from matplotlib.colors import ListedColormap

both.obs["is_pgc"] = (both.obs["fine_label"].astype(str) == PGC_LABEL).astype(float)

sc.pl.umap(
    both,
    color="is_pgc",
    cmap=ListedColormap(["lightgrey", "red"]),
    vmin=0, vmax=1,
    sort_order=True,          # draws the 1s last, so PGCs sit on top
    size=12,
    colorbar_loc=None,
    title=f"PGCs (n={int(both.obs['is_pgc'].sum())})",
)

## 7. Check the PGC labels before trusting them

The germline labels came from matching barcodes between two files. If that
worked, PGC labelled cells should express PGC markers. SOX17, NANOS3, POU5F1
and TFAP2C mark early PGCs. ++ ADD OLIVIA'S MARKERS

In [ ]:
markers = [g for g in ["SOX17", "NANOS3", "POU5F1", "TFAP2C", "PRDM1"]
           if g in both.var_names]
is_p = (both.obs["fine_label"].astype(str) == PGC_LABEL).to_numpy()
print(f"PGC labelled: {is_p.sum():,} cells, markers found: {markers}\n")

rows = []
for g in markers:
    x = np.asarray(both[:, g].X.todense()).ravel()
    rows.append({"gene": g,
                 "pct_in_PGC": round((x[is_p] > 0).mean() * 100, 1),
                 "pct_in_rest": round((x[~is_p] > 0).mean() * 100, 1)})
display(pd.DataFrame(rows).set_index("gene"))
print("the PGC column should be clearly higher for SOX17, NANOS3, POU5F1 and TFAP2C")

In [ ]:
both.obs["pgc_group"] = pd.Categorical(
    np.where(both.obs["fine_label"].astype(str) == PGC_LABEL, "PGC", "other"),
    categories=["PGC", "other"])

sc.pl.dotplot(
    both, markers, groupby="pgc_group",
    standard_scale="var",        # scales each gene 0 to 1 across the groups
    cmap="Reds",
    colorbar_title="scaled mean\nexpression",
    size_title="fraction of cells (%)",
    figsize=(len(markers) * 0.8 + 2, 2),
    show=False,
)
plt.savefig(os.path.join(OUTDIR, "dotplot_pgc_markers.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Germline coverage. lineageSomatic calls a cell GermCells when it is a germ
# cell that is not in human_germ, so those cells carry no fine germline label
# and would enter LIANA as a separate population alongside the PGCs.
GERM_LABELS = ["primordial germ cell", "oogonial cell", "oocyte", "germ cell",
               "male germ cell", "GermCells"]

sex_str = both.obs["sex"].astype(str)
groups = {"female": (sex_str == "female").to_numpy(),
          "male": (sex_str == "male").to_numpy(),
          "combined": np.ones(both.n_obs, dtype=bool)}

for name, m in groups.items():
    g = both.obs["fine_label"].astype(str)[m]
    found = {k: int((g == k).sum()) for k in GERM_LABELS if (g == k).any()}
    fine = sum(v for k, v in found.items() if k != "GermCells")
    coarse = found.get("GermCells", 0)
    print(f"{name}: {found}")
    if fine + coarse:
        print(f"   fine germline labels cover "
              f"{100 * fine / (fine + coarse):.0f}% of germ cells, "
              f"{coarse:,} left as GermCells")

# 'male germ cell' labels turn up in the female object, so check whether that
# comes from human_germ itself or from barcodes matching across sexes
mgc = germ.obs["cell_type"].astype(str) == "male germ cell"
if mgc.any():
    print("\nsex of the cells human_germ calls 'male germ cell':")
    print(germ.obs.loc[mgc, "sex"].value_counts().to_string())

`human_germ` itself has 532 male and 89 female cells labelled `male germ cell`.
The contradiction is in the published germ object, not in our matching.
One line in the methods covers it, and only 1 of those cells is in our window anyway.

76 cells in the combined object, out of 412 germ cells in the window. Worst case, if every one were a PGC, 327 becomes 403. The male verdict does not change either way: 14, or at most 16, against roughly 50 needed.

BUT 76 clears the 10 cell threshold, so GermCells will enter the LIANA run as its own cell type. If those cells are PGCs, the target population is split in two. THIS NEEDS TO BE CHECKED:

In [ ]:
grp = both.obs["fine_label"].astype(str)
sets = {"PGC": (grp == PGC_LABEL).to_numpy(),
        "GermCells": (grp == "GermCells").to_numpy(),
        "everything else": (~grp.isin([PGC_LABEL, "GermCells"])).to_numpy()}
print({k: int(v.sum()) for k, v in sets.items()})

rows = []
for g in markers:
    x = np.asarray(both[:, g].X.todense()).ravel()
    rows.append({"gene": g,
                 **{k: round(float((x[v] > 0).mean() * 100), 1)
                    for k, v in sets.items()}})
display(pd.DataFrame(rows).set_index("gene"))

In [ ]:
s = both.obs["fine_label"].astype(str)
both.obs["germ_group"] = pd.Categorical(
    np.select([s == PGC_LABEL, s == "GermCells"], ["PGC", "GermCells"],
              default="everything else"),
    categories=["PGC", "GermCells", "everything else"])
print(dict(both.obs["germ_group"].value_counts()))

sc.pl.dotplot(
    both, markers, groupby="germ_group",
    standard_scale="var", cmap="Reds",
    colorbar_title="scaled mean\nexpression",
    size_title="fraction of cells (%)",
    figsize=(len(markers) * 0.8 + 2, 2.4),
    show=False,
)
plt.savefig(os.path.join(OUTDIR, "dotplot_germcells_check.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Restricted to labels with at least 10 cells. standard_scale rescales each
# gene between the lowest and highest GROUP mean, so a 1 or 2 cell row can set
# an endpoint on sampling noise alone and squash the comparison that matters.
s = both.obs["fine_label"].astype(str)
GERM = [PGC_LABEL, "GermCells", "oogonial cell", "oocyte",
        "germ cell", "male germ cell"]
present = [g for g in GERM if (s == g).sum() >= 10]
print("shown:", present)
print("too few cells to plot:",
      {g: int((s == g).sum()) for g in GERM if 0 < (s == g).sum() < 10})

germ_only = both[s.isin(present).to_numpy()].copy()
germ_only.obs["fine_label"] = (germ_only.obs["fine_label"].astype(str)
                               .astype("category")
                               .cat.set_categories(present))
print(dict(germ_only.obs["fine_label"].value_counts()))

sc.pl.dotplot(
    germ_only, markers, groupby="fine_label",
    standard_scale="var", cmap="Reds",
    colorbar_title="scaled mean\nexpression",
    size_title="fraction of cells (%)",
    figsize=(len(markers) * 0.8 + 2, 3),
    show=False,
)
plt.savefig(os.path.join(OUTDIR, "dotplot_germ_labels.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Every germline label, including the ones too small for the plot above.
# No standard_scale here.
# Dot size is the fraction of cells expressing and is unaffected either way.
s = both.obs["fine_label"].astype(str)
GERM_ALL = [PGC_LABEL, "GermCells", "oogonial cell", "oocyte",
            "germ cell", "male germ cell"]
present_all = [g for g in GERM_ALL if (s == g).any()]
print({g: int((s == g).sum()) for g in present_all})
print("rows under 10 cells are noisy, read them as indicative only")

germ_all = both[s.isin(present_all).to_numpy()].copy()
germ_all.obs["fine_label"] = (germ_all.obs["fine_label"].astype(str)
                              .astype("category")
                              .cat.set_categories(present_all))

sc.pl.dotplot(
    germ_all, markers, groupby="fine_label",
    cmap="Reds",
    colorbar_title="mean expression\n(log normalised)",
    size_title="fraction of cells (%)",
    title="all germline labels",
    figsize=(len(markers) * 0.8 + 2, 3),
    show=False,
)
plt.savefig(os.path.join(OUTDIR, "dotplot_germ_labels_all.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Checks for the GermCells, to separate a real transitioning
# germ population from doublets or from cells that failed quality control.
# All of it reads data that has already been computed by the author.
lab = both.obs["fine_label"].astype(str)
gc = (lab == "GermCells").to_numpy()
pg = (lab == PGC_LABEL).to_numpy()
grp3 = pd.Series(np.select([pg, gc], ["PGC", "GermCells"], default="other"),
                 index=both.obs_names)
print(f"GermCells {int(gc.sum())}, PGCs {int(pg.sum())}\n")

# 1. positive for all three PGC specific markers at once
trio = [g for g in ["SOX17", "NANOS3", "TFAP2C"] if g in both.var_names]
pos = np.ones(both.n_obs, dtype=bool)
for g in trio:
    x = both[:, g].X
    x = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
    pos &= x > 0

print(f"1. positive for all of {trio}")
for nm, m in [("PGC", pg), ("GermCells", gc)]:
    print(f"     {nm:10s} {int(pos[m].sum()):4d} of {int(m.sum()):4d}  "
          f"({100 * pos[m].mean():.0f}%)")
print("   near the PGC rate  -> same cell state")
print("   near zero          -> moved past the PGC state")
print("   around half        -> a mixture, some are still PGCs\n")

# 2. where they sit in the clustering
ct = pd.crosstab(both.obs["leiden"], grp3)
ct = ct[[c for c in ["PGC", "GermCells"] if c in ct.columns] +
        [c for c in ["other"] if c in ct.columns]]
ct = ct.loc[(ct.get("PGC", 0) > 0) | (ct.get("GermCells", 0) > 0)]
print("2. clusters holding PGCs or GermCells")
print(ct.to_string())
print("   both in the same clusters -> one population on a transition")
print("   GermCells spread across somatic clusters -> doublet signature\n")

# 3. library size, which is what separates doublets
if "counts" in both.layers:
    C = both.layers["counts"]
    tot = np.asarray(C.sum(axis=1)).ravel()
    ngene = np.asarray((C > 0).sum(axis=1)).ravel()
    print("3. median library size per cell")
    for nm, m in [("PGC", pg), ("GermCells", gc), ("everything else", ~(pg | gc))]:
        print(f"     {nm:16s} {np.median(tot[m]):8.0f} counts, "
              f"{np.median(ngene[m]):6.0f} genes")
    print("   above BOTH the others  -> doublets")
    print("   close to the PGCs      -> the same population")
    print("   well below the PGCs    -> poorly captured, check high_mito below")

# 4. the quality flags the authors calculated
for col in ["low_ncounts", "high_mito", "low_ncounts_high_mito"]:
    if col in both.obs and both.obs[col].astype(str).nunique() > 1:
        t = pd.crosstab(grp3, both.obs[col].astype(str), normalize="index") * 100
        print(f"\n4. {col}, row percentages")
        print(t.round(1).to_string())

From `high_mito` seems like they are low quality PGCs (stressed or dying), and the authors seem to have treated them as such, since they excluded them from the re-analysis.
Also, they libraries are smaller than the PGCs.

In [ ]:
# Set this from the check above.
#   "keep"  leave GermCells as its own cell type, nothing changes
#   "merge" relabel them as PGCs
#   "drop"  remove them from the objects entirely
GERMCELLS_ACTION = "drop"   # <-- ALREADY CHECKED - LEAVE IT DROP

n_gc = int((both.obs["fine_label"].astype(str) == "GermCells").sum())

if GERMCELLS_ACTION == "keep":
    print(f"keeping {n_gc} GermCells as a separate cell type")

elif GERMCELLS_ACTION == "merge":
    lab = both.obs["fine_label"].astype(str)
    lab[lab == "GermCells"] = PGC_LABEL
    both.obs["fine_label"] = pd.Categorical(lab)
    print(f"merged {n_gc} GermCells into {PGC_LABEL!r}, "
          f"now {int((lab == PGC_LABEL).sum())} PGCs")

elif GERMCELLS_ACTION == "drop":
    # the neighbour graph was built before this, so it keeps edges for the
    # removed cells. Negligible at this size, but do not re-plot the UMAP
    # from the dropped object without recomputing.
    both = both[both.obs["fine_label"].astype(str) != "GermCells"].copy()
    print(f"dropped {n_gc} GermCells, {both.n_obs:,} cells left")

else:
    raise ValueError("GERMCELLS_ACTION must be 'keep', 'merge' or 'drop'")

# keep the plotting columns consistent with whatever was decided, since they
# are written into the h5ad files
lab = both.obs["fine_label"].astype(str)
both.obs["is_pgc"] = (lab == PGC_LABEL).astype(float)
both.obs["pgc_group"] = pd.Categorical(
    np.where(lab == PGC_LABEL, "PGC", "other"), categories=["PGC", "other"])
both.obs["germ_group"] = pd.Categorical(
    np.select([lab == PGC_LABEL, lab == "GermCells"], ["PGC", "GermCells"],
              default="everything else"),
    categories=["PGC", "GermCells", "everything else"])
print(f"PGCs now: {int(both.obs['is_pgc'].sum())}")

## 8. Write

Three files: female, male, and both sexes together.

Run LIANA on the female object first. That is the clean analysis, with enough
PGCs and several donors.
The male object almost certainly has too few PGCs to use on its own.

There is also the combined object that should be okay to use (CHECK with Gruhn).
Because `lineageSomatic` gives the ovary and the testis different names,
the sex specific senders stay separate in a combined run:
pre granulosa and Sertoli are two cell types, not one averaged one. Only the
PGCs pool, which is the point, and germ cells are still fairly similar between
the sexes at these stages.

The issue is the cell types that share a name across both sexes, endothelial
and the mesenchymal ones. Those would pool an ovary and a testis into one
sender. Say that in the write up if you use the combined object.

The neighbour graph and UMAP are removed from the two per sex files, because
slicing keeps the edges from the combined graph and that is not a graph
computed on the subset. The combined file keeps both. They were computed before
the 76 GermCells were removed, so the graph covers 76 more cells than the object
holds. Negligible at this size, but recompute if you re-plot from it.

In [ ]:
outputs = {
    "female": both[both.obs["sex"].astype(str) == "female"].copy(),
    "male": both[both.obs["sex"].astype(str) == "male"].copy(),
    "combined": both,
}

for name, sub in outputs.items():
    if name != "combined":
        # the graph and UMAP were built on all the cells, so a slice of them
        # is not a graph of the subset
        for k in ["connectivities", "distances"]:
            sub.obsp.pop(k, None)
        sub.uns.pop("neighbors", None)
        sub.obsm.pop("X_umap", None)

    pops = sub.obs["fine_label"].astype(str).value_counts()
    pops = pops[pops > 0]
    n_pgc = int(pops.get(PGC_LABEL, 0))

    p = os.path.join(OUTDIR, f"invivo_window_{name}.h5ad")
    sub.write_h5ad(p)
    print(f"{name}: {sub.n_obs:,} cells, {len(pops)} cell types, "
          f"{n_pgc:,} PGCs, {sub.obs['donor_id'].nunique()} donors -> {p}")

    thin = pops[pops < 10]
    if len(thin):
        print(f"   under 10 cells, drop these from LIANA: {dict(thin)}")
    if n_pgc < 50:
        print(f"   WARNING: {n_pgc} PGCs is too few for a ligand receptor "
              f"analysis in this object")

# which cell types exist in both sexes, and so would be pooled in a
# combined run
f_types = set(outputs["female"].obs["fine_label"].astype(str))
m_types = set(outputs["male"].obs["fine_label"].astype(str))
shared = sorted(f_types & m_types)
print(f"\ncell types present in both sexes, pooled if you use the combined "
      f"object: {shared}")
print(f"female only: {sorted(f_types - m_types)}")
print(f"male only:   {sorted(m_types - f_types)}")

## Notes for the write up

Set the LIANA cell type key to `fine_label` and the target to
`primordial germ cell`, with spaces.

Use the female file unless the combined one is also okay to use.

Counts are in `layers["counts"]`, log normalised values in `.X`.

The in vivo data is 10x 5' and in vitro data is (probably) 3'. That
changes which genes get detected, so trust clear present or absent differences
rather than small ones.

Most of the PGCs in this window sit at week 9. The 6 to 7 week part, which is
the closest to day 4 PGCLCs, has very few cells.

PGCs have 37,338 median counts against 7,622 for everything else, nearly five times.
LIANA scores interactions partly on the fraction of cells expressing a gene,
and that fraction rises with depth. So receptors on PGCs will look better expressed
than ligands on the senders, and interactions with PGCs as the receiver will be favoured
for a reason that is technical.
It also affects the in vitro comparison. A receptor that looks present in PGCs and
absent in PGCLCs could be depth rather than biology, and that stacks on top of the
5' against 3' difference.